In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Function Vectors Replication

## Goal
Replicate the core experiment from "Function Vectors in Large Language Models" (ICLR 2024) which demonstrates that:
1. Attention heads transport compact vector representations of ICL tasks ("function vectors")
2. These vectors can trigger task execution in zero-shot and shuffled-label contexts
3. Function vectors work across different prompting templates

## Overview
This notebook reimplements the key components without copying verbatim code:
1. Load model and dataset
2. Compute task-conditioned mean activations
3. Extract function vector using universal top heads
4. Evaluate on ICL, shuffled-label, zero-shot, and natural text contexts

In [2]:
# Setup: Import required packages and set up paths
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import Counter

# Set working directory
REPO_ROOT = '/net/scratch2/smallyan/function_vectors_eval'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# Check CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Repository root: {REPO_ROOT}")

# Disable gradient computation for inference
torch.set_grad_enabled(False)

# Set seeds for reproducibility
def set_all_seeds(seed=42):
    """Set seeds for reproducibility across all libraries"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)
print("Seeds set for reproducibility")

Using device: cuda
Repository root: /net/scratch2/smallyan/function_vectors_eval
Seeds set for reproducibility


## Part 1: Data Loading and Model Setup

We implement our own dataset loader and model configuration following the documented approach.

In [3]:
# Dataset class implementation (reimplemented from understanding of prompt_utils.py)
from sklearn.model_selection import train_test_split

class TaskDataset:
    """
    Dataset class for ICL task data with input-output pairs.
    Supports train/valid/test splitting.
    """
    def __init__(self, data):
        if isinstance(data, str):
            self.data = pd.read_json(data)
        elif isinstance(data, dict):
            self.data = pd.DataFrame(data)
        else:
            self.data = data
        self.data = self.data[['input', 'output']]
    
    def __getitem__(self, idx):
        if isinstance(idx, int):
            return self.data.iloc[idx].to_dict()
        elif isinstance(idx, (slice, list, np.ndarray)):
            return self.data.iloc[idx].to_dict(orient='list')
        elif isinstance(idx, str):
            return self.data[idx].tolist()
        raise ValueError(f"Invalid index type: {type(idx)}")
    
    def __len__(self):
        return len(self.data)

def load_task_dataset(task_name, data_dir='dataset_files', test_ratio=0.3, seed=32):
    """Load and split a task dataset"""
    # Find the dataset file
    for folder in ['abstractive', 'extractive']:
        path = os.path.join(data_dir, folder, f'{task_name}.json')
        if os.path.exists(path):
            dataset = TaskDataset(path)
            break
    else:
        raise FileNotFoundError(f"Dataset {task_name} not found")
    
    # Split into train/valid/test
    train_df, valid_df = train_test_split(dataset.data, test_size=test_ratio, random_state=seed)
    test_df, valid_df = train_test_split(valid_df, test_size=test_ratio, random_state=seed)
    
    return {
        'train': TaskDataset(train_df.to_dict(orient='list')),
        'valid': TaskDataset(valid_df.to_dict(orient='list')),
        'test': TaskDataset(test_df.to_dict(orient='list'))
    }

# Test dataset loading
dataset = load_task_dataset('antonym')
print(f"Antonym dataset loaded:")
print(f"  Train: {len(dataset['train'])} examples")
print(f"  Valid: {len(dataset['valid'])} examples")
print(f"  Test: {len(dataset['test'])} examples")
print(f"\nSample train pair: {dataset['train'][0]}")

Antonym dataset loaded:
  Train: 1678 examples
  Valid: 216 examples
  Test: 504 examples

Sample train pair: {'input': 'hardware', 'output': 'software'}


In [4]:
# Prompt construction utilities (reimplemented from understanding)

def build_prompt_data(word_pairs, query_pair=None, add_bos=True, shuffle_labels=False, prepend_space=True):
    """
    Construct prompt data dictionary from word pairs.
    
    Parameters:
    - word_pairs: dict with 'input' and 'output' lists
    - query_pair: dict with single 'input'/'output' for test query
    - add_bos: whether to prepend BOS token
    - shuffle_labels: whether to shuffle output labels
    - prepend_space: whether to add space before each word
    """
    prefixes = {'input': 'Q:', 'output': 'A:', 'instructions': ''}
    separators = {'input': '\n', 'output': '\n\n', 'instructions': ''}
    
    if add_bos:
        prefixes = {k: (v if k != 'instructions' else '<|endoftext|>' + v) for k, v in prefixes.items()}
    
    inputs = list(word_pairs.get('input', []))
    outputs = list(word_pairs.get('output', []))
    
    if shuffle_labels:
        outputs = np.random.permutation(outputs).tolist()
    
    if prepend_space:
        examples = [{'input': ' ' + str(i), 'output': ' ' + str(o)} for i, o in zip(inputs, outputs)]
        if query_pair:
            query_pair = {'input': ' ' + str(query_pair['input']), 'output': ' ' + str(query_pair['output'])}
    else:
        examples = [{'input': str(i), 'output': str(o)} for i, o in zip(inputs, outputs)]
    
    return {
        'instructions': '',
        'prefixes': prefixes,
        'separators': separators,
        'examples': examples,
        'query_target': query_pair
    }

def construct_prompt(prompt_data, query=None):
    """Build the full ICL prompt string"""
    if query is None and prompt_data['query_target']:
        query = prompt_data['query_target']['input']
    if isinstance(query, list):
        query = query[0]
    
    # Build primer from examples
    prompt = prompt_data['prefixes']['instructions'] + prompt_data['instructions'] + prompt_data['separators']['instructions']
    for ex in prompt_data['examples']:
        prompt += prompt_data['prefixes']['input'] + ex['input'] + prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output'] + ex['output'] + prompt_data['separators']['output']
    
    # Add query
    prompt += prompt_data['prefixes']['input'] + query + prompt_data['separators']['input']
    prompt += prompt_data['prefixes']['output']
    
    return prompt

# Test prompt construction
test_pairs = dataset['train'][:3]
test_query = dataset['test'][0]
prompt_data = build_prompt_data(test_pairs, query_pair=test_query)
prompt = construct_prompt(prompt_data)
print("Sample ICL prompt:")
print(repr(prompt[:300]) + "...")

Sample ICL prompt:
'<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: swift\nA:'...


In [5]:
# Load model and tokenizer (using GPT-J 6B as per demo)
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading GPT-J 6B model...")
model_name = 'EleutherAI/gpt-j-6b'

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True).to(device)
model.eval()

# Create model config based on architecture
MODEL_CONFIG = {
    'n_heads': model.config.n_head,
    'n_layers': model.config.n_layer,
    'resid_dim': model.config.n_embd,
    'name_or_path': model.config.name_or_path,
    'attn_hook_names': [f'transformer.h.{i}.attn.out_proj' for i in range(model.config.n_layer)],
    'layer_hook_names': [f'transformer.h.{i}' for i in range(model.config.n_layer)],
    'prepend_bos': False  # GPT-J doesn't auto-prepend BOS
}

print(f"\nModel loaded: {model_name}")
print(f"  Layers: {MODEL_CONFIG['n_layers']}")
print(f"  Heads: {MODEL_CONFIG['n_heads']}")
print(f"  Residual dim: {MODEL_CONFIG['resid_dim']}")
print(f"  Head dim: {MODEL_CONFIG['resid_dim'] // MODEL_CONFIG['n_heads']}")

Loading GPT-J 6B model...


OSError: PermissionError at /net/projects2/chai-lab/shared_models/hub/.locks/models--EleutherAI--gpt-j-6b/0e183edc2025ecfdba4429ba43c960224103b3c3dc26616503cdc2158a3d6c93.lock when downloading EleutherAI/gpt-j-6b. Check cache directory permissions. Common causes: 1) another user is downloading the same model (please wait); 2) a previous download was canceled and the lock file needs manual removal.

In [6]:
# Try loading from scratch directory or with a different cache
import os
os.environ['HF_HOME'] = '/net/scratch2/smallyan/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = '/net/scratch2/smallyan/.cache/huggingface'

from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading GPT-J 6B model from local cache...")
model_name = 'EleutherAI/gpt-j-6b'

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir='/net/scratch2/smallyan/.cache/huggingface')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    low_cpu_mem_usage=True,
    cache_dir='/net/scratch2/smallyan/.cache/huggingface'
).to(device)
model.eval()

# Create model config based on architecture
MODEL_CONFIG = {
    'n_heads': model.config.n_head,
    'n_layers': model.config.n_layer,
    'resid_dim': model.config.n_embd,
    'name_or_path': model.config.name_or_path,
    'attn_hook_names': [f'transformer.h.{i}.attn.out_proj' for i in range(model.config.n_layer)],
    'layer_hook_names': [f'transformer.h.{i}' for i in range(model.config.n_layer)],
    'prepend_bos': False  # GPT-J doesn't auto-prepend BOS
}

print(f"\nModel loaded: {model_name}")
print(f"  Layers: {MODEL_CONFIG['n_layers']}")
print(f"  Heads: {MODEL_CONFIG['n_heads']}")
print(f"  Residual dim: {MODEL_CONFIG['resid_dim']}")
print(f"  Head dim: {MODEL_CONFIG['resid_dim'] // MODEL_CONFIG['n_heads']}")

Loading GPT-J 6B model from local cache...


tokenizer_config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/930 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/24.2G [00:00<?, ?B/s]

In [7]:
# Check if model loaded successfully
print(f"Model loaded: {model_name}")
print(f"  Layers: {MODEL_CONFIG['n_layers']}")
print(f"  Heads: {MODEL_CONFIG['n_heads']}")
print(f"  Residual dim: {MODEL_CONFIG['resid_dim']}")
print(f"  Head dim: {MODEL_CONFIG['resid_dim'] // MODEL_CONFIG['n_heads']}")

In [8]:
# Verify model is loaded
try:
    print(f"Model: {model_name}")
    print(f"Model type: {type(model)}")
    print(f"Device: {model.device}")
    print(f"Layers: {MODEL_CONFIG['n_layers']}, Heads: {MODEL_CONFIG['n_heads']}")
except Exception as e:
    print(f"Error: {e}")

In [9]:
print("Testing...")